In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field


from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
load_dotenv(override=True)

True

In [2]:
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

nvidia_client = AsyncOpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)

nvidia_model = OpenAIChatCompletionsModel(
    model="nvidia/nemotron-3.5-super-vl-120b-a12b",
    openai_client = nvidia_client
)

In [3]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [4]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [5]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [6]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [7]:
checker = Agent(name = 'Checker', model=nvidia_model,
                 instructions="You review potential sales emails.", 
                output_type=EmailReview)

result = await Runner.run(checker, email)

[non-fatal] Tracing client error 401. Response data is redacted.


In [8]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=4, contains_placeholders=True)

In [9]:
review.is_professional

False

In [10]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered = is_problem)


In [11]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=nvidia_model, output_guardrails=[email_guardrail])

In [12]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

[non-fatal] Tracing client error 401. Response data is redacted.


In [13]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=nvidia_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

[non-fatal] Tracing client error 401. Response data is redacted.


**Subject:** Howdy [Name] – let’s tame that SOC2 beast 🤠

Howdy [Name],

I reckon you’ve got a lot on your plate running things over at [Company Name]. The last thing you need is a SOC2 audit breathing down your neck like a rattlesnake.

Most folks spend months drowning in spreadsheets and manual checks just to prove they’re secure. It’s a long, dusty trail that keeps you from actually growing your business.

At ComplAI, we built an AI-powered tool to do the heavy lifting for you. Think of us as your trusty AI sidekick for compliance. We help you wrangle your controls, automate the evidence gathering, and get you audit-ready faster than a draw at high noon.

No more late nights stressing over paperwork. Just a clean path to that SOC2 badge.

You got 10 minutes next Tuesday or Wednesday for a quick chat? I’d love to show you how we can clear the dust off your compliance process.

Saddle up,

[Your Name]
Sales Agent, ComplAI
[Your Calendar Link]
[Your Website]


[non-fatal] Tracing client error 401. Response data is redacted.


In [14]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

[non-fatal] Tracing client error 401. Response data is redacted.


Email is good


[non-fatal] Tracing client error 401. Response data is redacted.
